# SHIPP Stage 16 — Security Validation

**Owner:** Raneem  
**Priority:** P0  
**Purpose:** Read-only security validation for SHIPP.

This notebook validates the P0 security boundary without changing working pipeline or Agent behavior.

## Gates

- **16.1 Secret hygiene** — ORS uses Databricks Secrets in the existing Stage 31 implementation. No secret value is printed here.
- **16.2 PII / sensitive-data boundary** — verify downstream Silver/Gold/Search products do not expose prohibited sensitive columns.
- **16.3 Agent write controls** — reuse the already validated Stage 12/13 evidence; this notebook does **not** execute another write.


## Gate 16.1 — Secret hygiene

Verified implementation path in `notebooks/development/31_ors_route_enrichment.ipynb`:

```python
ors_api_key = dbutils.secrets.get(
    scope=ORS_SECRET_SCOPE,
    key=ORS_SECRET_KEY,
)
```

The ORS key is retrieved at runtime through Databricks Secrets and is not printed by this validation notebook.


In [0]:
# ============================================================
# STAGE 16 — PII / SENSITIVE DATA BOUNDARY VALIDATION
# READ ONLY
# ============================================================

print("=== STAGE 16 — PII BOUNDARY CHECK ===")

tables = [
    "bootcamp_students.shipp_silver.silver_listings",
    "bootcamp_students.shipp_silver.silver_requests",
    "bootcamp_students.shipp_gold.gold_candidate_matches",
    "bootcamp_students.shipp_gold.gold_listing_search_docs",
]

prohibited_columns = {
    "email",
    "email_address",
    "phone",
    "phone_number",
    "password",
    "password_hash",
    "api_key",
    "secret",
    "access_token",
    "refresh_token",
}

violations = []
checked_tables = []

for table_name in tables:
    print("\n----------------------------------------")
    print("TABLE:", table_name)

    if not spark.catalog.tableExists(table_name):
        print("SKIP — table does not exist")
        continue

    checked_tables.append(table_name)

    columns = {
        c.lower()
        for c in spark.table(table_name).columns
    }

    found = sorted(
        columns.intersection(prohibited_columns)
    )

    print("Sensitive columns found:", found)

    if found:
        violations.append(
            {
                "table": table_name,
                "columns": found,
            }
        )

assert checked_tables, (
    "BLOCKED — none of the expected downstream tables were available."
)

assert not violations, (
    f"FAIL — sensitive columns found downstream: {violations}"
)

print("\nChecked tables:", len(checked_tables))
print("PASS — no prohibited sensitive columns found in Silver/Gold/Search.")


## Gate 16.3 — Agent write controls

Do **not** rerun a write from this notebook. Reuse the validated Stage 12/13 evidence from `notebooks/validation/08_validate_agent_save_item.ipynb` and the Stage 13 controlled analytics proof.

Required controls already exercised in the validated Agent path:

- validate operational IDs;
- re-check current Lakebase listing state before write;
- reject duplicate saves;
- reject unavailable listings;
- write a valid `saved_items` record;
- record the action in `agent_activity`;
- confirm the result from the database.

This gate is evidence reuse, not a new implementation or another business write.
